In [1]:
# 1. Import the package
import os
import sys
module_path = os.path.abspath(os.path.join(r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src'))

if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
# 2. Create the test cluster
from dask_cluster_manager import DaskClusterManager

dask_cluster = DaskClusterManager()
dask_cluster.create_test_cluster()

In [3]:
# 3. If pulling data from Earth Engine, autheticate the Dask workers with Earth Engine
from dask_plugins import EEPlugin

json_key = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'
ee_plugin = EEPlugin(json_key)
dask_client = dask_cluster.get_dask_client
dask_client.register_plugin(ee_plugin)

{'tcp://127.0.0.1:54932': {'status': 'OK'}}

In [4]:
# 4. Create the parameters dictionary and the EarthEngineData object
from input_driver import EarthEngineData
import ee
import json

with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)

ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'map_function': prep_sr_l8
        }

earth_engine = EarthEngineData(parameters)
#chunk_size = {'time': 3, 'X': 9084, 'Y': 10578}
#chunk_size = {'time': 3, 'X': 1024, 'Y': 2048}
#earth_engine.chunk_dataset(chunk_size)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_0JLhFqfSY1uiEaW?source=Init


In [6]:
# 5. Run NDVI dataframe function to get optimal chunk size
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction(earth_engine)

# Apply the user-defined function
# If I load the dataset as a dask delayed, I don't need a template!
user_defined_func.tune_user_function(earth_engine, compute_ndvi)

Difference is GREATER than 1%
NEW SLICE: <xarray.Dataset> Size: 152B
Dimensions:  (time: 3, X: 2, Y: 2)
Coordinates:
  * time     (time) datetime64[ns] 24B 2020-12-29T18:57:32.281000 ... 2020-12...
  * X        (X) float64 16B -4.216e+05 -4.215e+05
  * Y        (Y) float64 16B -5.992e+05 -5.991e+05
Data variables:
    SR_B4    (time, X, Y) float32 48B dask.array<chunksize=(3, 2, 2), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 48B dask.array<chunksize=(3, 2, 2), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_2_bands:  SR_B7,SR_B5,SR_B3

In [7]:
# 6. Recreate a Dask cluster ready for full deployment.
dask_client.shutdown()

In [8]:
dask_cluster.create_local_cluster()

In [10]:
# 3. If pulling data from Earth Engine, autheticate the Dask workers with Earth Engine
dask_client = dask_cluster.get_dask_client
dask_client.register_plugin(ee_plugin)
earth_engine = EarthEngineData(parameters)

In [11]:
# 7. Do the full run!
result = user_defined_func.apply_user_function(earth_engine, compute_ndvi)

In [ ]:
# 8. Write results to a geotiff??